In [29]:
import pandas as pd
import numpy as np

In [55]:
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.compose import ColumnTransformer

In [31]:
df = pd.read_csv("Titanic-Dataset.csv",usecols=['Age','Fare','Survived'])

In [32]:
df.dropna(inplace=True)

In [33]:
df.shape

(714, 3)

In [34]:
df.head()

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500


In [35]:
X = df.iloc[:,1:]
y = df.iloc[:,0]

In [36]:
x_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [37]:
x_train.head(2)

,Age,Fare
328,31.0,20.5250
73,26.0,14.4542


In [38]:
clf = DecisionTreeClassifier()

In [39]:
clf.fit(x_train,y_train)
y_pred = clf.predict(X_test)

In [40]:
accuracy_score(y_test,y_pred)

0.6153846153846154

In [41]:
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

np.float64(0.6303012519561815)

In [42]:
kbin_age = KBinsDiscretizer(n_bins=5,encode='ordinal',strategy='kmeans')
kbin_fare = KBinsDiscretizer(n_bins=5,encode='ordinal',strategy='kmeans')

In [43]:
trf = ColumnTransformer([
    ('first',kbin_age,[0]),
    ('second',kbin_fare,[1])
])

In [44]:
x_train_trf = trf.fit_transform(x_train)
X_test_trf = trf.fit_transform(X_test)

In [45]:
trf.named_transformers_['first'].n_bins_

array([5])

In [46]:
trf.named_transformers_['first'].bin_edges_

array([array([ 1.        , 13.54505632, 25.49492263, 36.14664502, 48.33333333,
              62.        ])                                                   ],
      dtype=object)

In [47]:
output = pd.DataFrame({
    'age':x_train['Age'],
    'age_trf':x_train_trf[:,0],
    'fare':x_train['Fare'],
    'fare_trf':x_train_trf[:,1]
})

In [48]:
output['age_label'] = pd.cut(x = x_train['Age'],
    bins=trf.named_transformers_['first'].bin_edges_[0].tolist())

output['fare_label'] = pd.cut(x = x_train['Fare'],
    bins=trf.named_transformers_['second'].bin_edges_[0].tolist())

In [49]:
output.sample(5)

,age,age_trf,fare,fare_trf,age_label,fare_label
348,3.00,0.0,15.9000,0.0,"(1.0, 13.545]","(0.0, 35.744]"
124,54.00,3.0,77.2875,1.0,"(48.333, 62.0]","(35.744, 90.827]"
357,38.00,2.0,13.0000,0.0,"(36.147, 48.333]","(0.0, 35.744]"
276,45.00,3.0,7.7500,0.0,"(36.147, 48.333]","(0.0, 35.744]"
305,0.92,0.0,151.5500,2.0,NaN,"(90.827, 176.319]"


In [50]:
clf = DecisionTreeClassifier()
clf.fit(x_train_trf,y_train)
y_pred2 = clf.predict(X_test_trf)

In [51]:
accuracy_score(y_test,y_pred2)

0.6293706293706294

In [52]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

np.float64(0.630281690140845)

In [56]:
def discretize(bins,strategy):
    kbin_age = KBinsDiscretizer(n_bins=bins,encode='ordinal',strategy=strategy)
    kbin_fare = KBinsDiscretizer(n_bins=bins,encode='ordinal',strategy=strategy)

    trf = ColumnTransformer([
        ('first',kbin_age,[0]),
        ('second',kbin_fare,[1])
    ])

    X_trf = trf.fit_transform(X)
    print(np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy')))

    plt.figure(figsize=(14,4))
    plt.subplot(121)
    plt.hist(X['Age'])
    plt.title("Before")

    plt.subplot(122)
    plt.hist(X_trf[:,0],color='red')
    plt.title("After")
    
    plt.show()